# Affordability synthesis

Seventh package: joins the six sibling vectors into the affordability story:
best-available monthly income, contractual commitments (debt service + rent), the
invoiced-expenses floor, and the residual.

Key design points:
1. **Income basis is explicit** (`income_basis`): salary_net > ss_regular > irs_gross.
   Ratios on different bases are not comparable; the field says which is live.
2. **Rent is included**: RenterContractList contracts are tenant contracts, verified by
   matching `PersonalTaxData.Nif` in `RenterList` (135/135; note `PersonalData.NIF`
   uses a different encoding and does NOT match).
3. `residual_income = income - debt_service - rent - expenses_floor`; optimistic by the
   floor nature of invoiced expenses.
4. **Interpretation caveat**: the e-fatura floor can capture household-level spending
   (one member collects the invoices) while salary_net income is proponent-only; that
   asymmetry inflates burden for salaried couples. IRS-gross basis has the opposite
   issue (household income, gross). Read `income_basis` before comparing people.

In [ ]:
import json
import sys

import pandas as pd

sys.path.insert(0, ".")
from affordability_features import (affordability_features, proponent_profile,
                                    render_full_report, render_profile_text)

pd.set_option("display.width", 200)
raw = pd.read_csv("../data/data_full.csv")
snap = raw.drop_duplicates("uuid")[["uuid", "value"]].reset_index(drop=True)
feats = pd.DataFrame([
    {"uuid": u, **affordability_features(json.loads(v))}
    for u, v in zip(snap["uuid"], snap["value"])
])
print(f"{feats.shape[0]} proponents x {feats.shape[1] - 1} fields, "
      f"version {feats['features_version'].iloc[0]}")

## 1. Population affordability

In [ ]:
print("income basis:", feats["income_basis"].value_counts().to_dict())
KEY = ["monthly_income", "debt_service", "rent_monthly", "expenses_floor",
       "dsti", "burden_ratio", "residual_income"]
print(feats[KEY].describe(percentiles=[.1, .25, .5, .75, .9]).round(2).T.to_string())
print("\nnegative residual:", int((feats["residual_income"] < 0).sum()),
      "of", int(feats["residual_income"].notna().sum()), "computable")
print("\nby income basis (median residual):")
print(feats.groupby("income_basis")["residual_income"].agg(["median", "count"]).round(0).to_string())

## 2. The full report: one proponent, all seven sections

In [ ]:
fset = feats.set_index("uuid")
u = fset[(fset["income_basis"] == "salary_net")
         & (fset["residual_income"] > 0) & (fset["rent_monthly"].notna())].index[0]
vmap = dict(zip(snap["uuid"], snap["value"]))
print(render_full_report(json.loads(vmap[u])))

## 3. Validation: arithmetic identities and selection rules. Must print 0.

In [ ]:
mism = checked = 0
for uuid, v in zip(snap["uuid"], snap["value"]):
    p = json.loads(v)
    vecs = proponent_profile(p)
    a = vecs["affordability"]
    inc = vecs["incomes"]
    checks = []
    if a["monthly_income"] and a["residual_income"] is not None:
        recon = (a["monthly_income"] - (a["debt_service"] or 0)
                 - (a["rent_monthly"] or 0) - a["expenses_floor"])
        checks.append(abs(recon - a["residual_income"]) < 0.01)
        checks.append(abs((a["debt_service"] or 0) / a["monthly_income"] - a["dsti"]) < 0.001)
    expect_basis = ("salary_net" if inc["avg_net_salary_6m"] else
                    "ss_regular" if inc["avg_regular_income_6m"] else
                    "irs_gross" if inc["monthly_income_irs"] else "none")
    checks.append(a["income_basis"] == expect_basis)
    for ok in checks:
        checked += 1
        if not ok:
            mism += 1
            print("MISMATCH", uuid[:8])
print(f"{checked} checks, mismatches: {mism}")